In [55]:
import pandas as pd
import numpy as np
import awkward as ak
import matplotlib.pyplot as plt
import os
import h5py
import glob
import numpy as np
import yaml
import argparse
from matplotlib.backends.backend_pdf import PdfPages

In [56]:
input_file = '/global/cfs/cdirs/dune/www/data/2x2/nearline_run2/flowed_light/postrun_status_check/ACL_3000count_trig/mpd_run_data_rctl_1006'
START_p = 0
TICKS = 400
SAVE = True
pdf_location = './Dead_Alive_Test'
ADC_LIST = [0,1,2,3,4,5,6,7]

In [57]:
## define system constants ##
SAMPLE_RATE = 0.016 # us
BIT = 4  # factor from unused ADC bits on LRS: would be nice to have in a resource .yaml
PRE_NOISE = 50 # Will need to be re-defined once we know beam timing
SAT = 32767
EXPECTED_RATE = 60 # In Hz, obviously depends on subrun conditions
RATE_THD = 20 # Hz over expected trigger rate considered allowable
RANGE = 2000 # ADC range allowed in DQM for baslines
    
SIPM_CHANNELS = ([4,5,6,7,8,9] + \
                 [10,11,12,13,14,15] + \
                 [20,21,22,23,24,25] + \
                 [26,27,28,29,30,31] + \
                 [36,37,38,39,40,41] + \
                 [42,43,44,45,46,47] + \
                 [52,53,54,55,56,57] + \
                 [58,59,60,61,62,63])

In [58]:
## To read metadata ##

def visit_func(name, node) :
    if 'ref' in name:
        return
    if [x for x in node.attrs.keys()] != []:
        print ('\nFull object pathname is:', node.name)
    for v in node.attrs.items():
        print(v)
#file.visititems(visit_func)

In [59]:
def thd_correct(array):
    
    array_shape = np.shape(array)[-1]
    num_bins = int((array_shape // 25)+1)
    indices = np.arange(0, num_bins) * 25  # (39,)
    start_indices, end_indices = indices[:-1], indices[1:]  # (39,)

    segment_range = np.arange(25)  # Shape: (25,)
    index_array = start_indices[:, None] + segment_range  # Shape: (39, 25)

    # Extract data using advanced indexing
    sliced_data = array[..., index_array]

    ranges = np.abs(np.ptp(sliced_data, axis=-1))
    #ranges = np.ptp(sliced_data, axis=-1)  # Compute range (n, 8, 64, 39)
    means = np.mean(sliced_data, axis=-1)  # Compute mean (n, 8, 64, 39)

    # Find ordering based on the smallest range
    smallest_ordering = np.argsort(ranges, axis=-1)  # Shape (n, 8, 64, 39)
    # Mask zero ranges
    mask_zero = (ranges != 0)
    ranges = np.where(mask_zero, ranges, np.nan)
    means = np.where(mask_zero, means, np.nan)

    # Sort means using the ordering
    sorted_means = np.take_along_axis(means, smallest_ordering, axis=-1)  # Shape (n, 8, 64, 39)
    sorted_range = np.take_along_axis(ranges, smallest_ordering, axis=-1)
    # Compute average of 2nd, 3rd, and 4th smallest means
    average_mean = np.mean(sorted_means[..., 1:3], axis=-1)  # Shape (n, 8, 64)
    expanded_mean = average_mean[..., None] 
    broadcasted_mean = np.tile(expanded_mean, (1, array_shape))  
    filtered_wvfm = array - broadcasted_mean

    return filtered_wvfm

In [60]:
def tag_dark_counts(N_Files, START=0, INPUT_name=None, adc_list=None, ticks=600):

     file_num = 0
     dark_count_wvfm = np.zeros((8,64,ticks), dtype=np.int64)
     peak_value = np.zeros((8,64), dtype=np.int64)
     SIPM_CHANNELS = ([4,5,6,7,8,9] + \
                     [10,11,12,13,14,15] + \
                     [20,21,22,23,24,25] + \
                     [26,27,28,29,30,31] + \
                     [36,37,38,39,40,41] + \
                     [42,43,44,45,46,47] + \
                     [52,53,54,55,56,57] + \
                     [58,59,60,61,62,63])

     for Nf in range(N_Files):
          file = f'{INPUT_name}_p{START+file_num}.FLOW.hdf5'

          if file_num < N_Files:
               print('file number:', file_num)
               file_num += 1
               if not os.path.isfile(file):
                    print('NOT A FILE')
                    continue
               else:
                    with h5py.File(file, 'r') as h5:

                         size_bytes = os.path.getsize(file)
                         size_gb = size_bytes / (1024 ** 3)
                         print(f"File size: {size_gb:.2f} GB")

                         MULT = int(size_gb // 3)
                         if MULT < 1:
                              MULT = 1
                              print(f"Using MULT = {MULT} for data reduction.")

                         offbeam_wvfm_v1 =  h5['light/wvfm/data']['samples'][::MULT,:,:,:]
                         offbeam_wvfm_v2 =  thd_correct(offbeam_wvfm_v1)/4
                         del offbeam_wvfm_v1
                         if adc_list is None:
                              adc_list = [0,1,2,3,4,5,6,7]
                         for adc in adc_list:
                              for ch in SIPM_CHANNELS:
                                   #mask = np.where(np.ptp(offbeam_wvfm_v2[:, adc, ch, :], axis=-1) < 1e3)[0]
                                   mask = np.argmax(np.max(offbeam_wvfm_v2[:, adc, ch, :], axis=-1))
                                   #if len(mask) == 0:
                                   #     del mask
                                   #     continue
                                   #else:
                                   #max_idx = np.argmax(np.max(offbeam_wvfm_v2[mask, adc, ch, :], axis=-1))
                                   max_value = np.max(offbeam_wvfm_v2[mask, adc, ch, :], axis=-1)
                                   #print('adc:', adc, ' ch:', ch, ' max_value:', max_value)
                                   if max_value > peak_value[adc, ch]:
                                        #new_val = (-1*peak_value[adc, ch]) + max_value
                                        peak_value[adc, ch] *= 0
                                        peak_value[adc, ch] += max_value
                                        dark_count_wvfm[adc, ch, :] *= 0
                                        dark_count_wvfm[adc, ch, :] += np.array(offbeam_wvfm_v2[mask, adc, ch, :], dtype=np.int16)
                                        del mask
                                        del max_value
                                        #del new_val
                                   else:
                                        del mask
                                        del max_value 
                         del offbeam_wvfm_v2
     return dark_count_wvfm, peak_value

In [61]:
def plot_waveforms(adc_list=None, sipm_channels=None, peak_value=None, signal_output=None, pdf_output=None, save_bool=False):
    dead_mask = np.zeros((8,64), dtype=int)
    if adc_list is None:
        adc_list = [0,1,2,3,4,5,6,7]
    for adc in adc_list:
        for ch in sipm_channels:
            fig, ax = plt.subplots(figsize=(8,2))
            ax.set_title(f'ADC {adc} Channel {ch}')
            noise_check = (np.ptp(signal_output[adc, ch, :], axis=-1)*2)
            #mask = np.where(np.max(signal_output[:, adc, ch, :], axis=-1) > 600)[0]
            if (peak_value[adc, ch]*4) < 300:
                if (peak_value[adc, ch]*4) < (noise_check*1.5):
                    textstr = f'No Max Value Detected'
                    props = dict(boxstyle='round', facecolor='white', edgecolor='crimson')
                    ax.text(
                        0.5, 0.25, textstr,
                        transform=ax.transAxes,
                        fontsize=10,
                        verticalalignment='top',
                        horizontalalignment='center',
                        bbox=props
                    )
                elif np.mean(signal_output[adc, ch, :]) < 4:
                    textstr = f'No Max Value Detected'
                    props = dict(boxstyle='round', facecolor='white', edgecolor='crimson')
                    ax.text(
                        0.5, 0.25, textstr,
                        transform=ax.transAxes,
                        fontsize=10,
                        verticalalignment='top',
                        horizontalalignment='center',
                        bbox=props
                    )
                else: 
                    dead_mask[adc, ch] += 1
                    
            elif (peak_value[adc, ch]*4) < (noise_check*1.5):
                textstr = f'No Max Value Detected'
                props = dict(boxstyle='round', facecolor='white', edgecolor='crimson')
                ax.text(
                    0.5, 0.25, textstr,
                    transform=ax.transAxes,
                    fontsize=10,
                    verticalalignment='top',
                    horizontalalignment='center',
                    bbox=props
                )

            else: 
                dead_mask[adc, ch] += 1
            ax.axhline(300, color='green', linestyle='--', alpha=0.8)
            ax.axhline(-300, color='orange', linestyle='--', alpha=0.8)
            ax.plot(signal_output[adc, ch, :]*4, color='blue', alpha=0.8)
            ax.set_xlabel('Time Sample [0.016 μs]')
            #plt.ylim((-100, 800))
            ax.set_ylabel('ADC Counts')
            plt.grid()
            plt.tight_layout()
            if save_bool==True:
                pdf_output.savefig(fig, dpi=150)
            plt.close(fig)
            np.savez(os.path.join(pdf_location, 'Dead_Alive_channel_mask.npz'), dead_mask=dead_mask)

In [62]:
SIGNAL_OUTPUT, PEAK_VALUES = tag_dark_counts(N_Files=1, START=START_p, INPUT_name=input_file, adc_list=ADC_LIST, ticks=TICKS)

file number: 0
File size: 1.13 GB
Using MULT = 1 for data reduction.


In [63]:

#for adc in range(8):
if SAVE==True:

    with PdfPages(f"{pdf_location}/DeadAlive_Check.pdf", "w") as pdf:
       
       plot_waveforms(adc_list=ADC_LIST, sipm_channels=SIPM_CHANNELS, peak_value=PEAK_VALUES, signal_output=SIGNAL_OUTPUT, pdf_output=pdf, save_bool=SAVE)

/tmp/ipykernel_2166287/3981982019.py:4: MatplotlibDeprecationWarning: The 'keep_empty' parameter of __init__() was deprecated in Matplotlib 3.10 and will be removed in 3.12. This parameter does nothing. If any parameter follows 'keep_empty', they should be passed as keyword, not positionally.
  with PdfPages(f"{pdf_location}/DeadAlive_Check.pdf", "w") as pdf:
